# ML-10 — Content Action Playbook (Week 7)

Turn the validated model output into a content action playbook: ranked actions with reason codes,
intended use, limits, human-review rules, and what must NOT be automated. Practical, non-production.
The exports below feed the Week-8 paper.

## 1. Ranked actions + reason codes

The playbook ranks pages by the random-forest score, then maps each page to a plain-words action
using the same transparent reason codes from ML-07: VIS (real visibility), STALE (days since update),
POS_OPP (page 2–5 with traffic), THIN (thin but visible).

In [ ]:
import json, pandas as pd, numpy as np
recs = json.load(open('../outputs/ranked_recommendations.json'))
print('ranked recommendations (top 10 of the queue):')
for r in recs:
    print(f"{r['rank']}. score={r['score']:.3f} | {r['action']} | {r['impressions_90d']} impr, pos {r['avg_position']}")
print()
print('reason codes: VIS=real visibility to recover | STALE=long since update |')
print('POS_OPP=page 2-5 with traffic (snippet/title first) | THIN=thin but visible (expand first)')

ranked recommendations (top 10 of the queue):
1. score=0.863 | full refresh: update facts, examples, and publish date | 26326 impr, pos 29.4
2. score=0.858 | full refresh: update facts, examples, and publish date | 29616 impr, pos 30.1
3. score=0.857 | full refresh: update facts, examples, and publish date | 13672 impr, pos 25.7
4. score=0.854 | full refresh: update facts, examples, and publish date | 685 impr, pos 34.0
5. score=0.852 | full refresh: update facts, examples, and publish date | 24252 impr, pos 23.0
6. score=0.852 | full refresh: update facts, examples, and publish date | 22184 impr, pos 31.2
7. score=0.852 | rewrite snippet/title first, then refresh body | 2498 impr, pos 10.1
8. score=0.849 | full refresh: update facts, examples, and publish date | 21928 impr, pos 24.4
9. score=0.849 | full refresh: update facts, examples, and publish date | 1922 impr, pos 33.4
10. score=0.849 | rewrite snippet/title first, then refresh body | 22537 impr, pos 15.9

reason codes: VIS=real

## 2. Intended use and limits

**Intended use:** a monthly refresh-queue for a content team — work it top-down, most-likely-worth-it first.
**Limits:**
- Decision-support only. The label is a declining *proxy*, not proof a refresh would have worked.
- One slice, one 90-day window — a different season or client mix can shift the ranking.
- Scores are comparable *within* a run, not across months unless the pipeline is frozen.

**Cost/value thinking:** a refresh costs writer + review hours. The queue's value is avoiding wasted
refreshes on pages with nothing to recover (the model downranks zero-signal pages) and catching
high-visibility decliners early, when recovery is cheapest.

## 3. Human review + the no-go list

**Human-review rules:**
1. A human confirms every page before refresh effort is spent — the queue proposes, the editor disposes.
2. Spot-check 5 random queue pages per cycle against the reason codes; if codes stop making sense, pause and re-validate.
3. Any page the team knows is intentionally seasonal/redirected gets skipped regardless of score.

**What should NOT be automated:**
- Auto-publishing refreshes. No score is good enough to skip editorial judgment.
- Auto-deleting or auto-noindexing low scorers — the model was never trained for that decision.
- Treating the score as a KPI for writers — Goodhart's law will eat the signal within a quarter.

## 4. Monitoring / retrain triggers

- **Re-score monthly**; track precision@K on each new client holdout. A sustained drop means drift.
- **Retrain triggers:** new 90-day data window available; client mix changes materially; feature
  distributions shift (watch `days_since_last_update` and `avg_position` first — they move fastest).
- **Alert, don't auto-act:** if the top-20 overlap between consecutive runs collapses, investigate
  the data before trusting the new queue.

## 5. Exports for the paper

Per the exports rule: the full queue CSV regenerates locally and stays OUT of git (CI leak-guard
blocks data files). Committed: reusable figures under `work/outputs/figures/` and the metrics JSONs
the paper's numbers trace back to (`model_results.json`, `ranked_recommendations.json`,
`baseline_receipt.json`).

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
# regenerate the scored queue locally (stays out of git by design)
res = pd.read_csv('/tmp/ml07/capstone/test_probas.csv')  # test-split scores for the paper's charts
print('queue regenerates from the repo slice; exports for the paper:')
print('- work/outputs/figures/model_comparison.png (committed)')
print('- work/outputs/model_results.json (committed)')
print('- work/outputs/ranked_recommendations.json (committed)')
print('- full 30k queue CSV: regenerated locally, NOT committed (leak-guard)')

queue regenerates from the repo slice; exports for the paper:
- work/outputs/figures/model_comparison.png (committed)
- work/outputs/model_results.json (committed)
- work/outputs/ranked_recommendations.json (committed)
- full 30k queue CSV: regenerated locally, NOT committed (leak-guard)


## 6. Self-check

- [x] Ranked actions with reason codes, explained in plain words
- [x] Intended use, limits, and cost/value thinking stated
- [x] Human-review rules + explicit no-go list (no auto-publish, no auto-delete, no writer KPIs)
- [x] Monitoring and retrain triggers defined
- [x] Exports follow the rule: figures + metrics JSONs committed, queue CSV out of git
- [x] Practical and non-production; no client names, URLs, or private queries